In [ ]:
%load_ext autoreload
%autoreload 2

from datetime import datetime, date
from okx.store import OrderbookStore, populate, FEATURES
import polars as pl

# Initialize the store
store = OrderbookStore(
    data_root="data/okx",
    manifest_path="data/okx/manifest.sqlite"
)

In [ ]:
# Populate BTC-USD-SWAP data for a few days
populate(
    store,
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 8, 1),
    end=datetime(2025, 10, 1),
    verbose=True,
    max_workers=4
)

In [ ]:
# Store deletion example
# Example: Delete raw orderbook data for a specific family/type/date

# Let's delete one day's data for BTC-USD FUTURES (e.g. August 15, 2025)
delete_date = date(2025, 8, 15)

# store.delete_raw('BTC-USD', 'FUTURES', delete_date) (uncomment to delete)

# You can check if it's deleted from the manifest:
have_after = store.manifest.have('BTC-USD', 'FUTURES', delete_date, 'raw')
print(f"Raw for {delete_date} present after delete? {have_after}")


In [ ]:
lf = store.get(
    inst_type='FUTURES',
    inst_family='BTC-USD',
    start=datetime(2024, 6, 1),
    end=datetime(2024, 6, 2),
    depth=0 
)

# Collect and display
df = lf.collect()

In [ ]:
print(f"Shape: {df.shape}")
df.head()

In [ ]:
print(df['symbol'].unique())

In [ ]:
from utils import parse_option_name

# Since df is a Polars DataFrame (not a lazyframe at this point, since .collect() was called earlier), proceed accordingly.

# Analyze null count and percentage of each column
num_rows = df.height
nulls = df.null_count()
# Polars DataFrame returned by null_count() is a DataFrame, not a dict
# Convert to dict: use to_dict(as_series=False)[column_name][0]
nulls_dict = {col: nulls.select(col).to_series()[0] for col in df.columns}
nulls_pct = {col: (count / num_rows) * 100 for col, count in nulls_dict.items()}

# Display as a sorted table for readability, with both count and percentage
print("Null count and percentage per column:")
print(f"{'Column':<20} {'Nulls':>10} {'Pct':>8}")
for col in sorted(df.columns):
    count = nulls_dict[col]
    pct = nulls_pct[col]
    print(f"{col:<20} {count:10d} {pct:7.2f}%")

# Compute all unique option combos using parse_option_name
unique_symbols = df['symbol'].unique()
combos = set()
for sym in unique_symbols:
    # skip null/None values if present
    if sym is None:
        continue
    try:
        parsed = parse_option_name(sym)
        combo = (parsed[0], parsed[1], parsed[2])  # e.g. ('BTC-USD', expiry_datetime, strike)
        combos.add(combo)
    except Exception as e:
        print(f"Could not parse symbol '{sym}': {e}")

print("\nAll unique option combos (underlying, expiry, strike):")
for combo in sorted(combos, key=lambda x: (x[0], x[1], x[2])):
    print(combo)



In [ ]:
print(len(df))

In [ ]:
futures_df = store.get(
    inst_type='FUTURES',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 10, 1),
    depth=0 
).collect()

In [ ]:
swap_df = store.get(
    inst_type='SWAP',
    inst_family='BTC-USD',
    start=datetime(2025, 9, 1),
    end=datetime(2025, 10, 1),
    depth=0 
).collect()

In [ ]:
unique_times_futures = set(futures_df['timeMs'].to_list())
unique_times_swap = set(swap_df['timeMs'].to_list())

In [ ]:
print(unique_times_futures - unique_times_swap)

In [ ]:
print(swap_df.shape)

In [ ]:
print(317425202 - 189060720)

In [ ]:
print(futures_df.head())

In [ ]:
class TestClass:
    def __init__(self, name: str):
        self.name = name

    def __str__(self):
        return f"TestClass: {self.name}"


In [ ]:
c = TestClass("test")
c.name = "test2"
print(c)